# Zero-ridge evidence

Compare two in-distribution alignments, $F=\tilde F=0.1,0.4$: optimal ridge versus aspect ratio, $R(\lambda)-R(0)$ at $r=10$, and the right derivative at zero versus $F$. Only total excess risk is displayed.

Run all cells to load the included arrays, redraw the figure and save its PDF. Set `RECOMPUTE=True` to rerun the numerical stage. The exact zero endpoint is included; a positive right derivative is local boundary evidence, and a finite search cap does not certify infinite ridge.


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import inspect
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from copy import deepcopy

# Use only this distribution's local utilities, caches and output directories.
BUNDLE_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p/'dms'/'config.py').is_file()
                    and (p/'notebooks'/'05_zero_ridge.ipynb').is_file()), None)
if BUNDLE_ROOT is None:
    raise RuntimeError('Start the kernel in the reproduction folder or its notebooks/ directory.')
foreign = [name for name, module in sys.modules.items()
           if (name == 'dms' or name.startswith('dms.'))
           and getattr(module, '__file__', None)
           and not Path(module.__file__).resolve().is_relative_to(BUNDLE_ROOT/'dms')]
if foreign:
    raise RuntimeError('Restart the kernel: another checkout of dms is already imported.')
sys.path.insert(0, str(BUNDLE_ROOT))
PROJECT_ROOT = BUNDLE_ROOT  # retained renderer name; always local to this bundle
OUTPUT_DIR = BUNDLE_ROOT/'output'
importlib.invalidate_caches()
for module_name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory', 'dms.optimization', 'dms.ridge_diagnostics', 'dms.result_cache', 'dms.infinity_diagnostics', 'dms.appendix_evidence', 'dms.zero_ridge_evidence'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)
from dms.config import DMSConfig
from dms.result_cache import ResultCache
from dms.zero_ridge_evidence import cached_zero_evidence
plt.rcParams.update({'axes.labelsize':11, 'axes.labelweight':'bold',
                     'xtick.labelsize':11, 'ytick.labelsize':11, 'legend.fontsize':11})
figures = {}

from tools.portable_results import load_results, assert_settings_match

# False: reproduce from the included arrays. True: calculate from the inputs below.
RECOMPUTE = False


In [ ]:
cfg = DMSConfig(
    q=20, sigma=1.0, f_train=0.1, f_test=0.1, eta=0.0,
    signal_norm=10.0, F=0.4, F_test=0.4,
    F_neq=0.0, C_neq=0.0, M_tot=0.0, M_neq=0.0, M_neq_test=0.0,
)
R_VALUES = np.geomspace(0.01, 100.0, 81)
ALIGNMENTS = np.linspace(0.0, 1.0, 101)
EXACT_SEARCH = dict(lambda_max=1e3, n_grid=161)
EXACT_COLOR = "#243247"

# In-distribution signal fractions and the aspect ratio shared by panels b/c.
ZERO_F_VALUES = [0.1, 0.4]
ZERO_R_CHECK = 10.0  # shared by the dashed r guide, risk cut and slope panel
# Middle panel: evaluated ridge range and resolution.
ZERO_LAMBDA_RANGE = (0.0, 0.2)
ZERO_LAMBDA_POINTS = 201
ZERO_LAMBDA_GRID = np.linspace(*ZERO_LAMBDA_RANGE, ZERO_LAMBDA_POINTS)
# Independent optimization domain/resolution; not the middle-panel display range.
ZERO_SEARCH = dict(EXACT_SEARCH)  # currently lambda_max=1e3, n_grid=161
# ZERO_SEARCH.update(lambda_max=1e3)  # edit only to change the search cap

ZERO_INPUTS = dict(config=cfg.as_dict(), r_check=ZERO_R_CHECK, F_values=list(ZERO_F_VALUES),
    near_zero_lambdas=ZERO_LAMBDA_GRID.copy(), r_values=R_VALUES.copy(),
    alignments=ALIGNMENTS.copy(), search=dict(ZERO_SEARCH))
ZERO_CACHE_OPTIONS = dict(directory=BUNDLE_ROOT/'.cache'/'zero_ridge',
                          enabled=True, verbose=True)
ZERO_FORCE_COMPUTE = False


## Compute exact endpoint, slopes and finite-cap optima

`ZERO_LAMBDA_RANGE` and `ZERO_LAMBDA_POINTS` define the middle-panel computation.
Its display limits are controlled separately by `ZERO_PLOT['xlims'][1]`.
`ZERO_SEARCH` sets the independent optimization cap and resolution. The refined
minimum is added to the stored cut so its marker lies on the curve. The $F$ slope
grid extends to 1 even when the plotted range ends at 0.6.

In [ ]:
if RECOMPUTE:
    ZERO_EVIDENCE, ZERO_EVIDENCE_ID = cached_zero_evidence(
        ResultCache(**ZERO_CACHE_OPTIONS), ZERO_INPUTS, compute=True, force=ZERO_FORCE_COMPUTE)
else:
    published, metadata = load_results(BUNDLE_ROOT/'data'/'zero_ridge', restore_records=True)
    assert_settings_match(metadata['settings'], ZERO_INPUTS)
    ZERO_EVIDENCE, ZERO_EVIDENCE_ID = published['evidence'], published['identity']
ZERO_LOADED_INPUTS = deepcopy(ZERO_INPUTS)


## Editable renderer and presentation controls
Plot construction stays visible here. Changing styles does not invalidate numerical results.

In [ ]:
def draw_zero_evidence(data, style):
    fig, axes = plt.subplots(1,3,figsize=(style['width'],style['height']),
                             dpi=style['dpi'],layout='constrained')
    fig.get_layout_engine().set(**style['layout_padding'])
    colors = plt.get_cmap(style['F_cmap'])(np.linspace(*style['F_color_span'],len(data['curves'])))
    for (F,row),color in zip(data['curves'].items(),colors):
        options = dict(color=color,linewidth=style['linewidth'],linestyle='-') | style['F_styles'].get(F,{})
        color = options['color']
        label = style['F_labels'].get(F,fr'$F={F:g}$')
        optimum = row['optimum']
        axes[0].plot(optimum.r_values,optimum.ridge_lambdas,label=label,**options)
        axes[1].plot(row['lambdas'],row['delta'],label=label,**options)
        if style['show_zero_markers']:
            zero = optimum.status == 'zero'
            # Empty unclipped marker artists have a canvas-origin bounding box
            # and can incorrectly enlarge constrained-layout margins.
            if np.any(zero):
                axes[0].plot(optimum.r_values[zero],np.zeros(zero.sum()),
                             **(dict(color=color) | style['zero_markers']))
        if style['show_cap_markers']:
            upper = optimum.status == 'upper'
            if np.any(upper):
                axes[0].plot(optimum.r_values[upper],optimum.ridge_lambdas[upper],
                             **(dict(color=color) | style['cap_markers']))
        if style['show_minima']:
            marker = style['cap_markers'] if row['minimum_status']=='upper' else style['minimum_marker']
            axes[1].plot([row['minimum_lambda']],[row['minimum_delta']],
                         **(dict(color=color) | marker))
            if row['minimum_status']=='upper':
                print(f'F={F:g}: cut optimum hits the finite search cap; shown as a hollow triangle.')
            for limits,value in ((style['xlims'][1],row['minimum_lambda']),
                                 (style['ylims'][1],row['minimum_delta'])):
                if limits is not None and ((limits[0] is not None and value < limits[0])
                                           or (limits[1] is not None and value > limits[1])):
                    print(f'F={F:g}: the cut minimum lies outside the explicit plot limits.')
                    break
        if style['show_F_guides']:
            axes[2].axvline(F,**(dict(style['F_guide']) | dict(color=color)))
    axes[0].set_xscale('log')
    axes[0].set_yscale('symlog',linthresh=style['ridge_linthresh'],
                       linscale=style['ridge_linscale'])
    axes[0].set_ylim(bottom=0)
    if style['show_r_guide']:
        axes[0].axvline(data['r_check'],**style['r_guide'])
    slope_x, slope_y = np.asarray(data['alignments']), np.asarray(data['slopes'])
    if style['slope_auto_y_to_xlim'] and style['xlims'][2] is not None:
        # Crop only the plotted path; the computed F sweep stays unchanged.
        # Retain the adjacent samples so lines reach non-sampled window edges.
        lo, hi = style['xlims'][2]
        first = 0 if lo is None else max(0, np.searchsorted(slope_x, lo, side='right')-1)
        last = len(slope_x) if hi is None else min(len(slope_x), np.searchsorted(slope_x, hi)+1)
        slope_x, slope_y = slope_x[first:last], slope_y[first:last]
    axes[2].plot(slope_x,slope_y,**style['slope'])
    if style['show_zero_guides']:
        for ax in axes[1:]: ax.axhline(0,**style['zero_guide'])
    for i,ax in enumerate(axes):
        ax.set(xlabel=style['xlabels'][i],ylabel=style['ylabels'][i],title=style['titles'][i])
        for which in ('x','y'):
            limits,ticks = style[which+'lims'][i],style[which+'ticks'][i]
            if limits is not None: getattr(ax,'set_'+which+'lim')(limits)
            if ticks is not None: getattr(ax,'set_'+which+'ticks')(ticks)
        ax.tick_params(labelsize=style['tick_size'])
        for label in (ax.xaxis.label,ax.yaxis.label):
            label.set_fontsize(style['label_size']);label.set_fontweight('bold')
        ax.title.set_fontsize(style['title_size'])
        ax.grid(style['grid'])
        if style['panel_labels']:
            ax.text(*style['panel_label_xy'],chr(97+i),transform=ax.transAxes,
                    fontsize=style['panel_label_size'],fontweight='bold')
    if style['show_legend']:
        if style['legend_panel'] not in (0,1):
            raise ValueError('Put the single F legend in panel 0 or 1.')
        axes[style['legend_panel']].legend(**style['legend'])
    for i,text in enumerate(style['annotations']):
        if text is False: continue
        if text is None: text = fr'$r={data["r_check"]:g}$' if i > 0 else ''
        if text:
            axes[i].text(*style['annotation_xy'],text,transform=axes[i].transAxes,
                         ha='left',va='top',fontsize=style['label_size'])
    return fig

def plot_zero_evidence(style, *, force=False):
    assert_settings_match(ZERO_LOADED_INPUTS, ZERO_INPUTS)
    data, identity = ZERO_EVIDENCE, ZERO_EVIDENCE_ID
    try:
        implementation = [inspect.getsource(f) for f in (draw_zero_evidence, plot_zero_evidence)]
    except (OSError, TypeError):
        implementation, force = [], True
    fig, _ = ResultCache(**ZERO_CACHE_OPTIONS).get('appendix_zero_figure',
        dict(data=identity,style=style,rcparams={k:repr(v) for k,v in plt.rcParams.items()}),
        lambda: draw_zero_evidence(data,style), implementation=implementation,force=force)
    figures['appendix_zero_ridge'] = fig
    display(fig); plt.close(fig)
    return fig


In [ ]:
# Plot-only controls. These panels have different quantities/units, hence independent y axes.
ZERO_PLOT = dict(
    width=10.,height=3,dpi=140,label_size=11,tick_size=11,title_size=11,
    panel_labels=True,panel_label_xy=(-.10,1.03),panel_label_size=12,
    layout_padding=dict(w_pad=.03,h_pad=.03,wspace=.08,hspace=.03),
    titles=['','',''],grid=False,
    xlabels=[r'Aspect ratio $r=qL/n$',r'Ridge $\lambda$',r'Matched signal $F=\tilde F$'],
    ylabels=[r'Optimal ridge $\lambda_{\rm opt}$',r'Risk difference $R(\lambda)-R(0)$',
             r'Risk slope $\left.\frac{\partial R}{\partial\lambda}\right|_{\lambda=0^+}$'],
    # Entries are [left, middle, right]; None keeps automatic limits.
    xlims=[None,None,(0.,.6)],ylims=[None,None,None],
    # Example middle-panel zoom: set xlims[1] to (0., .05); no recomputation.
    xticks=[None,None,None],yticks=[None,None,None],
    ridge_linthresh=1e-4,  # linear region ends at the first positive decade tick
    ridge_linscale=.35,   # shrink the space from zero to 1e-4
    slope_auto_y_to_xlim=True,  # fit the right y range to the visible F interval
    F_cmap='cividis',F_color_span=(.15,.85),F_styles={},F_labels={},linewidth=1.8,
    slope=dict(color=EXACT_COLOR,linewidth=1.8,linestyle='-'),
    show_r_guide=True,r_guide=dict(color='0.4',linestyle='--',linewidth=1.),
    show_F_guides=True,F_guide=dict(linestyle='--',linewidth=1.,alpha=.9),
    show_minima=True,
    minimum_marker=dict(marker='o',markersize=5,markerfacecolor='white',
                        markeredgewidth=1.2,linestyle='None',zorder=5,clip_on=False),
    show_zero_markers=True,
    zero_markers=dict(marker='o',markersize=3,linestyle='None',clip_on=False),
    show_cap_markers=True,
    cap_markers=dict(marker='^',markerfacecolor='none',markersize=4,linestyle='None'),
    show_zero_guides=True,zero_guide=dict(color='0.6',linewidth=.8,linestyle=':'),
    show_legend=True,legend_panel=0,legend=dict(loc='best',fontsize=11,frameon=True),
    annotations=[False,None,None],annotation_xy=(.04,.96),  # None: stored inputs; False: hide
)
ZERO_FORCE_REDRAW = False


## Draw the figure

This cell uses the loaded numerical arrays and the editable controls above.


In [ ]:
zero_paper_fig = plot_zero_evidence(ZERO_PLOT, force=ZERO_FORCE_REDRAW)


## Export the displayed figure
Saving does not evaluate theory, optimize or redraw.

In [ ]:
PDF_FIGURES = {'appendix_zero_ridge': 'appendix_zero_ridge.pdf'}
PDF_OUTPUT_DIR = OUTPUT_DIR
missing = [key for key in PDF_FIGURES if key not in figures]
if missing: raise KeyError(f'Display these figures first: {missing}')
PDF_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for key,filename in PDF_FIGURES.items():
    output = PDF_OUTPUT_DIR/Path(filename).with_suffix('.pdf')
    figures[key].savefig(output,format='pdf',bbox_inches='tight')
    print(output)
